# NLP Explorer · Live coding

Hasta ahora seguimos el texto con dibujos: piezas, IDs y representaciones numéricas. ¿Ocurre así con un modelo real? Vamos a comprobarlo y a comparar dos maneras de pedir lo mismo.

**15 minutos · Google Colab · CPU.** Ejecuta las celdas de arriba abajo.

**Notebook guiado.** Completa los bloques `# Live coding: completa esta parte` con las pistas de cada sección; la solución contiene el código completo.

## 0. Las herramientas para comprobarlo

Para pasar de los dibujos al cálculo necesitamos Sentence Transformers, una librería que permite cargar modelos y obtener vectores de frases. Instalaremos la librería y usaremos un modelo ya entrenado, sin entrenarlo aquí ni llamar a una API comercial de embeddings. Los cálculos se harán en el runtime de Colab.

Ejecuta la instalación y espera a que termine antes de seguir. Se necesita conexión para instalar y descargar el modelo; no necesitas GPU, Drive ni una clave de API. Usamos `sentence-transformers==6.1.0`.

In [ ]:
%pip install -q sentence-transformers==6.1.0

## 1. Un modelo para nuestras frases · 2 min

Ya tenemos las herramientas; necesitamos un modelo que transforme frases en vectores. `MODEL_NAME` identifica el que usaremos: «paraphrase» alude a formulaciones distintas de una idea, «multilingual» a varios idiomas, incluido el español, y MiniLM a un modelo compacto. Lo cargaremos en CPU y aprovecharemos la descarga para recordar la pregunta: ¿relacionará dos maneras de pedir lo mismo?

Carga `model` con `SentenceTransformer(MODEL_NAME, device="cpu")` y muestra `model.device` para comprobar el dispositivo.

In [ ]:
from sentence_transformers import SentenceTransformer, util

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Live coding: completa esta parte
# Carga model con SentenceTransformer en CPU y muestra model.device.

## 2. ¿Cómo ve realmente nuestras palabras? · 4 min

En la web afirmamos que el texto se divide en piezas. ¿Lo hace realmente nuestro modelo, y reutiliza piezas entre «perro» y «perritos»? Consultemos su tokenizador antes de pedirle representaciones.

1. Accede al tokenizador con `tokenizer = model.tokenizer`.
2. Recorre `words` con `for word in words:`.
3. Obtén `tokens` con `tokenizer.tokenize(word)` y sus `ids` con `tokenizer.convert_tokens_to_ids(tokens)`.
4. Muestra texto, tokens e IDs con `print`.

Observamos las piezas sin añadir los tokens especiales del procesamiento completo. **`▁` no es `_`:** aquí marca aproximadamente comienzo de palabra o espacio previo; otros tokenizadores pueden usar otra convención.

In [ ]:
words = ["perro", "perros", "perrito", "perritos", "antidesestabilización"]

# Live coding: completa esta parte
# Accede al tokenizador, recorre words y muestra texto, tokens e IDs.

## 3. ¿El tokenizador entiende el contexto? · 2 min

Ya vemos piezas e identificadores, pero queda la pregunta de «banco»: ¿cambiarán entre el parque y retirar dinero? Comparemos ambas frases para distinguir identificar una pieza de interpretar su uso. El token es una pieza, su ID es una entrada del vocabulario y un embedding es un vector de números.

Tokeniza las dos frases y muestra tokens e IDs. Recorre `zip(tokens, ids)` para emparejar cada pieza con su identificador; con `if token == "▁banco"` muestra solo `token, token_id` de la pieza que queremos comparar.

In [ ]:
bank_sentences = [
    "Me senté en un banco del parque",
    "Fui al banco a retirar dinero",
]

# Live coding: completa esta parte
# Tokeniza cada frase y muestra tokens e IDs.
# Recorre sus parejas y muestra la pieza ▁banco con su ID.

La misma pieza y el mismo ID aparecen en ambas frases: el tokenizador ha dividido e identificado, pero no ha resuelto el significado. Para incorporar el contexto necesitamos el procesamiento del modelo. Aquí solo inspeccionamos tokens e IDs; no extraemos representaciones internas de «banco».

## 4. De tokens a una frase completa · 3 min

Hasta ahora inspeccionamos la entrada; ahora necesitamos representar mensajes completos. A pide cambiar una contraseña y B modificar una clave de acceso, mientras C habla de lluvia. Esperamos que A y B estén más relacionadas: obtengamos sus vectores para poder comprobarlo.

1. Usa `embeddings = model.encode(sentences, convert_to_tensor=True)` para obtener los vectores como tensor, una estructura numérica.
2. Muestra `embeddings.shape` para ver su forma.
3. Muestra `embeddings[0, :8]` para ver ocho valores del primer vector.

In [ ]:
A = "Quiero cambiar mi contraseña"
B = "Necesito modificar mi clave de acceso"
C = "Mañana va a llover"
sentences = [A, B, C]

# Live coding: completa esta parte
# Genera embeddings como tensor, muestra su forma y ocho valores del primero.

Al ver `torch.Size([3, 384])`, ¿qué representa primero el **3**? Hay tres frases, una por fila; **384** es la cantidad de valores por frase que produce este modelo, no una lista de conceptos con nombre. Los ocho valores mostrados hacen tangible el vector, aunque no necesitamos interpretar cada componente por separado.

## 5. Tenemos vectores. ¿Cómo los comparamos? · 2 min

Mirar 384 números frente a otros 384 no basta: necesitamos una regla de comparación. Recuperemos la dirección de las flechas de la web; la similitud del coseno compara la orientación de los vectores. Calculemos A con B y A con C para contrastar nuestra expectativa.

- `embeddings[0]`, `[1]` y `[2]` corresponden a A, B y C.
- `util.cos_sim` calcula la similitud y `.item()` extrae el único resultado como número.
- Guarda `similarity_ab` y `similarity_ac` y muéstralos con `:.4f`, cuatro decimales.

La puntuación no es un porcentaje ni una probabilidad.

In [ ]:
# Live coding: completa esta parte
# Calcula A ↔ B y A ↔ C con util.cos_sim y extrae cada número con .item().
# Muestra similarity_ab y similarity_ac con cuatro decimales.

## 6. ¿Qué hemos conseguido? · 1 min

¿Qué comparación ha salido mayor y cómo encaja con lo que dicen las frases? Esperamos A ↔ B considerablemente mayor que A ↔ C: podemos relacionar formulaciones distintas sin exigir las mismas palabras. Un valor cercano a cero, incluso ligeramente negativo, no demuestra antonimia.

Comparar representaciones puede servir para buscar y organizar información, pero una puntuación de similitud no demuestra que un texto sea verdadero ni que responda a una pregunta.

## 7. Ahora repetid vosotros el recorrido · 1 min

Partimos del texto, comprobamos sus piezas e IDs, obtuvimos vectores de frases y usamos una regla para compararlos. ¿Podéis repetir el recorrido con otras frases y explicar el resultado? En el reto haréis esa comprobación y diréis también qué no permite concluir la puntuación.